# Bharat-EO-Bench v1 · T1 — FROZEN-ENCODER PROBE (the clean head-to-head)

Full fine-tune gave a **tie** (Prithvi-300 ~= ImageNet U-Net) confounded by the
decoder and by run-to-run noise. This isolates the actual question — **is an
EO-pretrained representation better than an ImageNet one?** — by:
 - **freezing** each encoder (train only the SAME decoder on all) -> no decoder
   confound, and 600M fits a T4 (no encoder optimizer states);
 - comparing **same-architecture ViTs**: Prithvi-300, Prithvi-600, ImageNet ViT;
 - **3 seeds** -> mean ± std (so a real gap is distinguishable from noise).

Kaggle: upload `bharat-eo-chips-t1.zip`, GPU T4, Internet ON, + Add Input, Run All.


In [7]:
import importlib.metadata as md
KEEP = {p: md.version(p) for p in ("numpy","pandas","scipy","scikit-learn")}
!pip install -q segmentation-models-pytorch timm
!pip install -q terratorch
pins=" ".join(f"{k}=={v}" for k,v in KEEP.items())
!pip install -q --no-deps --force-reinstall {pins}
!python -c "import numpy;print('numpy',numpy.__version__)"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
numpy 2.0.2


In [8]:
import os, glob, random, json, numpy as np, pandas as pd, torch, torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, jaccard_score, accuracy_score
dev = "cuda" if torch.cuda.is_available() else "cpu"; print("device", dev)
ENCODERS = ["prithvi_eo_v2_300", "prithvi_eo_v2_600", "imagenet_vit"]
IN_BANDS, EPOCHS, K, SEEDS = 6, 25, 11, [0, 1, 2]


device cuda


In [9]:
# --- robust chip locate: extracted dir OR auto-extract a .zip OR find manifest -
import zipfile
def _find_npy():
    for pat in ("/kaggle/input/**/img/*.npy","/kaggle/working/**/img/*.npy","data/chips_v1/img/*.npy","data/chips/img/*.npy"):
        h=glob.glob(pat,recursive=True)
        if h: return h
    return []
imgs=_find_npy()
if not imgs:                                          # maybe the dataset is an un-extracted .zip
    for z in glob.glob("/kaggle/input/**/*.zip",recursive=True):
        try:
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith(".npy") for n in zf.namelist()[:3000]):
                    print("extracting",z,"..."); zf.extractall("/kaggle/working/chips")
        except Exception as e: print("  zip skip",os.path.basename(z),str(e)[:50])
    imgs=_find_npy()
if not imgs:                                          # last resort: locate via manifest.csv
    for m in glob.glob("/kaggle/input/**/manifest.csv",recursive=True)+glob.glob("/kaggle/working/**/manifest.csv",recursive=True):
        r=os.path.dirname(m); h=glob.glob(f"{r}/img/*.npy")
        if h: imgs=h; break
if not imgs:
    print("!! No chips found. Contents of /kaggle/input:")
    for p in glob.glob("/kaggle/input/*"): print("   ",p,"->",[os.path.basename(x) for x in glob.glob(p+'/*')][:8])
    raise SystemExit("Attach the chips dataset: Notebook -> '+ Add Input' -> add 'bharat-eo-chips-t1' (or upload the .zip), then Run All.")
ROOT=os.path.dirname(os.path.dirname(imgs[0])); print("root",ROOT,"| n=",len(imgs))
mp=os.path.join(ROOT,"manifest.csv")
if os.path.exists(mp):
    man=pd.read_csv(mp)
else:                                                 # no manifest -> deterministic 70/15/15 fallback
    ch=sorted(os.path.basename(p)[:-4] for p in imgs); r=np.random.RandomState(0).permutation(len(ch))
    sp=np.where(r<0.7*len(ch),"train",np.where(r<0.85*len(ch),"val","test"))
    man=pd.DataFrame({"chip":ch,"split":sp}); print("  [no manifest] using random 70/15/15 split")
if "split" not in man.columns: man["split"]="train"
print("  splits:",man.split.value_counts().to_dict())
def load_img(c): return np.transpose(np.load(f"{ROOT}/img/{c}.npy").astype(np.float32),(2,0,1))[:IN_BANDS]
tr_chips = man.loc[man.split=="train","chip"].tolist()
va_chips = man.loc[man.split.isin(["val","test"]),"chip"].tolist()
st=np.stack([load_img(c) for c in tr_chips[:200]]); MU=st.mean((0,2,3),keepdims=True)[0]; SD=st.std((0,2,3),keepdims=True)[0]+1e-6
cnt=np.zeros(K)
for c in tr_chips:
    l=np.load(f"{ROOT}/lbl/{c}.npy").ravel(); l=l[l!=255]; cnt+=np.bincount(l,minlength=K)
CW=torch.tensor(np.clip(cnt.sum()/(K*np.clip(cnt,1,None)),0.5,8.0).astype(np.float32))
class DS(torch.utils.data.Dataset):
    def __init__(s,ch,tr=False): s.ch,s.tr=ch,tr
    def __len__(s): return len(s.ch)
    def __getitem__(s,i):
        c=s.ch[i]; x=(load_img(c)-MU)/SD; y=np.load(f"{ROOT}/lbl/{c}.npy").astype(np.int64)
        if s.tr:
            if random.random()<.5: x=x[:,:,::-1]; y=y[:,::-1]
            k=random.randint(0,3); x=np.rot90(x,k,(1,2)); y=np.rot90(y,k)
        return torch.from_numpy(x.copy()).float(), torch.from_numpy(y.copy()).long()


root /kaggle/input/datasets/crakashpreetham/bharat-e0-chips-t1 | n= 5271
  splits: {'train': 3741, 'test': 806, 'val': 724}


In [10]:
# --- frozen encoders (all ViT) + a shared trainable decoder ------------------
class UpDecoder(nn.Module):
    def __init__(s,D,k):
        super().__init__(); ch=[D,256,128,64,32]; s.b=nn.ModuleList()
        for i in range(4): s.b.append(nn.Sequential(nn.Conv2d(ch[i],ch[i+1],3,padding=1),
            nn.GroupNorm(8,ch[i+1]),nn.GELU(),nn.Upsample(scale_factor=2,mode="bilinear",align_corners=False)))
        s.c=nn.Conv2d(ch[4],k,1)
    def forward(s,f):
        for b in s.b: f=b(f)
        return s.c(f)

def make_encoder(name):
    """returns (encoder_module, feat_fn) where feat_fn(x)->[B,D,h,w]; encoder frozen."""
    if name=="imagenet_vit":
        import timm
        enc=timm.create_model("vit_base_patch16_224",pretrained=True,in_chans=IN_BANDS,num_classes=0).to(dev)
        def ff(x):
            f=enc.forward_features(x)
            if f.dim()==3:
                f=f[:,1:,:]; B,N,D=f.shape; hw=int(round(N**.5)); f=f.transpose(1,2).reshape(B,D,hw,hw)
            return f
    else:
        from terratorch import BACKBONE_REGISTRY
        enc=BACKBONE_REGISTRY.build(name,pretrained=True,num_frames=1,
            bands=["BLUE","GREEN","RED","NIR_NARROW","SWIR_1","SWIR_2"]).to(dev)
        def ff(x):
            try: f=enc(x)
            except Exception: f=enc(x.unsqueeze(2))
            if isinstance(f,(list,tuple)): f=f[-1]
            if f.dim()==3:
                B,N,Dd=f.shape; hw=int(round(N**.5))
                if hw*hw!=N: f=f[:,1:,:]; N=f.shape[1]; hw=int(round(N**.5))
                f=f.transpose(1,2).reshape(B,Dd,hw,hw)
            return f
    for p in enc.parameters(): p.requires_grad=False
    enc.eval()
    with torch.no_grad(): D=ff(torch.zeros(1,IN_BANDS,224,224,device=dev)).shape[1]
    return enc, ff, D

def dice(lg,t,ig=255,eps=1.):
    m=(t!=ig); tt=t.clone(); tt[~m]=0
    oh=F.one_hot(tt,lg.shape[1]).permute(0,3,1,2).float()*m.unsqueeze(1)
    p=lg.softmax(1)*m.unsqueeze(1); inter=(p*oh).sum((0,2,3)); un=p.sum((0,2,3))+oh.sum((0,2,3))
    return 1-((2*inter+eps)/(un+eps)).mean()

def probe(name, seed):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    enc, ff, D = make_encoder(name)
    dec = UpDecoder(D, K).to(dev)
    opt=torch.optim.AdamW(dec.parameters(),lr=1e-3,weight_decay=1e-4)
    sc=torch.amp.GradScaler("cuda",enabled=(dev=="cuda")); ce=nn.CrossEntropyLoss(weight=CW.to(dev),ignore_index=255)
    trdl=torch.utils.data.DataLoader(DS(tr_chips,True),batch_size=8,shuffle=True,num_workers=2,drop_last=True)
    vadl=torch.utils.data.DataLoader(DS(va_chips),batch_size=8,num_workers=2)
    for ep in range(EPOCHS):
        dec.train()
        for x,y in trdl:
            x,y=x.to(dev),y.to(dev); opt.zero_grad()
            with torch.no_grad(): f=ff(x)                 # frozen features (no encoder grad/opt)
            with torch.amp.autocast("cuda",enabled=(dev=="cuda")):
                lg=F.interpolate(dec(f),(224,224),mode="bilinear",align_corners=False); loss=ce(lg,y)+dice(lg,y)
            sc.scale(loss).backward(); sc.step(opt); sc.update()
    dec.eval(); P,T=[],[]
    with torch.no_grad():
        for x,y in vadl:
            f=ff(x.to(dev)); lg=F.interpolate(dec(f),(224,224),mode="bilinear",align_corners=False)
            p=lg.argmax(1).cpu().numpy().ravel(); t=y.numpy().ravel(); k=t!=255; P.append(p[k]); T.append(t[k])
    P,T=np.concatenate(P),np.concatenate(T)
    del enc,dec; torch.cuda.empty_cache()
    return f1_score(T,P,average="macro",zero_division=0), jaccard_score(T,P,average="macro",zero_division=0)


In [ ]:
# --- run: each frozen encoder x 3 seeds -> mean +/- std ----------------------
res={}
for name in ENCODERS:
    fs=[]
    for s in SEEDS:
        try:
            mf1,miou=probe(name,s); fs.append((mf1,miou)); print(f"  {name} seed{s}: macroF1 {mf1:.3f} mIoU {miou:.3f}")
        except Exception as e:
            print(f"  [skip] {name} seed{s}: {str(e).splitlines()[0][:70]}")
    if fs:
        a=np.array(fs); res[name]={"mf1":[float(a[:,0].mean()),float(a[:,0].std())],"miou":[float(a[:,1].mean()),float(a[:,1].std())]}
print("\n=== FROZEN-PROBE head-to-head (encoder representation quality) ===")
print(f"  {'encoder':<20} {'macro-F1 (mean±std)':<22} {'mIoU (mean±std)'}")
for n,r in sorted(res.items(),key=lambda x:-x[1]['mf1'][0]):
    print(f"  {n:<20} {r['mf1'][0]:.3f} ± {r['mf1'][1]:.3f}        {r['miou'][0]:.3f} ± {r['miou'][1]:.3f}")
json.dump(res, open("probe_headtohead.json","w"), indent=2)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Prithvi_EO_V2_300M.pt:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

  prithvi_eo_v2_300 seed0: macroF1 0.562 mIoU 0.425
  prithvi_eo_v2_300 seed1: macroF1 0.566 mIoU 0.432
  prithvi_eo_v2_300 seed2: macroF1 0.540 mIoU 0.400


Prithvi_EO_V2_600M.pt:   0%|          | 0.00/2.64G [00:00<?, ?B/s]

  prithvi_eo_v2_600 seed0: macroF1 0.565 mIoU 0.425


## Reading it
- Same frozen protocol + same decoder on all -> the ONLY difference is pretraining.
- If Prithvi (EO) > ImageNet ViT by **more than the ±std**, EO pretraining genuinely
  helps for India LC. If within ±std, it's a tie (the full-finetune result).
- Prithvi-600 vs 300 (both fit now, frozen) answers whether FM scale helps.
